# Training on Google Colab

Everything in this project already runs on a laptop CPU. The reason to come
here is **speed**: the runs still outstanding take hours on a CPU and minutes
on a GPU.

| Run | Laptop CPU | Colab GPU |
|---|---|---|
| U-Net to convergence | ~3 h | ~5 min |
| Swin to convergence | ~5 h | ~10 min |
| 3 seeds x 2 models | ~20 h | ~45 min |

**Before running anything: Runtime -> Change runtime type -> T4 GPU.**
Without that you are on a CPU slower than your own laptop.

The datasets are **not** uploaded. They are regenerated here from `seed 0`,
which takes about four seconds and produces files identical to the ones on
your machine. Only the code needs to travel.

## 1. Check the GPU is actually there

In [ ]:
import torch

print("torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU, then run again."
    )

## 2. Get the code here

The repository is private, so a plain `git clone` will not work. Two options.

**Option A, simplest.** On your laptop, zip the project folder (right-click
the `qmri-t1-benchmark` folder, "Send to -> Compressed folder"). Run the cell
below and pick that zip. You do **not** need to include `data/` or
`outputs/` -- the data is regenerated in step 3.

**Option B.** Make the repository public for a few minutes and
`!git clone https://github.com/Rabaaab/qmri-t1-benchmark.git` instead.

In [ ]:
import zipfile, pathlib, os
from google.colab import files

uploaded = files.upload()  # pick your zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall("/content/project")

# Locate the project root by its pyproject.toml, whatever the zip
# nested it under. Anchoring on a folder name instead is a trap:
# rglob('src/qmri').parent gives src/, not the root.
root = next(pathlib.Path("/content/project").rglob("pyproject.toml")).parent
os.chdir(root)
print("working in:", root)
print(sorted(p.name for p in root.iterdir()))
assert (root / "scripts").is_dir(), "scripts/ not found -- wrong folder"


## 3. Rebuild the datasets

Reproducible from the seed, so these are byte-identical to the files on your
laptop. Nothing had to be uploaded.

In [ ]:
!python scripts/build_dataset.py

## 4. The classical baseline

This one does **not** benefit from a GPU -- it is tens of thousands of tiny
independent optimisations on the CPU. Colab gives you more cores than your
laptop, so it is still faster, but expect several minutes.

If you already have `data/baseline_test.npz` from your laptop (2.4 MB),
upload it instead and skip this cell.

In [ ]:
!python scripts/fit_baseline.py --split test

## 5. Train to real convergence

**This is the whole point of coming here.**

On the laptop both networks stopped because they hit the epoch cap, not
because they stopped improving -- U-Net at 189/200, Swin at 95/100. That
matters: an earlier U-Net stopped at 40 epochs appeared to erase 38% of
small lesions, and trained properly it was 13%.

The limits below are deliberately generous. **Check the printed
"best epoch" afterwards: if it is well below the maximum, training ended
because it plateaued, and the result can be trusted.** If it is near the
maximum again, raise `--epochs` and run it once more.

In [ ]:
!python scripts/train_mlp.py --epochs 400 --patience 40 --device cuda

In [ ]:
!python scripts/train_spatial.py --arch unet --depth 3 --base-channels 16 \
    --epochs 800 --patience 60 --device cuda

In [ ]:
!python scripts/train_spatial.py --arch swin --depth 2 --embed-dim 32 \
    --num-heads 4 --window 8 --patch-size 2 \
    --epochs 600 --patience 60 --device cuda

## 6. Repeat across seeds

One run is an anecdote. Three let you say whether the difference between the
models is real or just luck. Each checkpoint is saved under its own name so
nothing is overwritten.

Only run this once step 5 shows genuine convergence -- otherwise you are
repeating an unconverged result three times.

In [ ]:
import subprocess, shutil, pathlib, os, sys

# Fail loudly. check=True alone raises without showing stderr, so a
# broken run reports only an exit code and you cannot tell why.
assert pathlib.Path("scripts/train_spatial.py").exists(), (
    f"wrong directory: {os.getcwd()}"
)

for seed in (1, 2):
    for arch, extra in [
        ("unet", ["--depth", "3", "--base-channels", "16", "--epochs", "800"]),
        ("swin", ["--depth", "2", "--embed-dim", "32", "--epochs", "600"]),
    ]:
        print(f"
=== {arch}, seed {seed} ===", flush=True)
        result = subprocess.run(
            [sys.executable, "scripts/train_spatial.py", "--arch", arch,
             "--patience", "60", "--device", "cuda", "--seed", str(seed)] + extra,
            capture_output=True, text=True,
        )
        print("
".join(result.stdout.splitlines()[-12:]))
        if result.returncode != 0:
            print("FAILED, exit", result.returncode)
            print(result.stderr[-3000:])
            raise SystemExit("stopping: see the error above")
        depth = extra[1]
        src = pathlib.Path(f"outputs/{arch}_d{depth}.pt")
        kept = src.with_name(f"{arch}_d{depth}_seed{seed}.pt")
        shutil.copy(src, kept)
        print("saved as", kept.name)


## 7. Score everything and make the figures

In [ ]:
!python scripts/record_results.py
!python scripts/make_figures.py

## 8. Bring the results home

**Do this before closing the tab.** Colab deletes everything when the runtime
disconnects, and it disconnects on its own after a while.

This downloads the trained models, the figures and the results database --
a few megabytes. The datasets are not included because they regenerate from
the seed.

In [ ]:
import shutil, pathlib
from google.colab import files

out = pathlib.Path("/content/results")
out.mkdir(exist_ok=True)
for pattern, folder in [
    ("outputs/*.pt", "outputs"),
    ("figures/*.png", "figures"),
    ("data/results.sqlite", "data"),
    ("data/baseline_test.npz", "data"),
]:
    target = out / folder
    target.mkdir(parents=True, exist_ok=True)
    for path in pathlib.Path(".").glob(pattern):
        shutil.copy(path, target / path.name)

archive = shutil.make_archive("/content/qmri_results", "zip", out)
print("size:", round(pathlib.Path(archive).stat().st_size / 1024**2, 1), "MB")
files.download(archive)

## 9. Back on your laptop

Unzip into the project folder, overwriting `outputs/`, `figures/` and the two
files in `data/`. Then regenerate the tables locally so the README and the
database match the new runs:

```
python scripts/record_results.py
python scripts/make_figures.py
```

Commit the figures (they are tracked); `outputs/` and `data/` stay
gitignored, since they are regenerated rather than stored.